# Explorando Self-Attention — el corazón del Transformer

Vamos a implementar self-attention paso a paso, igual que hicimos con BPE.
Después formalizamos en `src/transformer/self_attention.py`.

**Plan:**
1. Setup: tensor de ejemplo (4 tokens, 8-dim embeddings)
2. Crear las matrices W_q, W_k, W_v (learnable)
3. Computar Q, K, V
4. Attention scores: Q @ K^T
5. Scaling + softmax
6. Weighted sum: attn @ V
7. Output: las mismas 4 posiciones, ahora contextualizadas

In [1]:
import torch

# Simulemos 4 tokens ya embedidos (en la práctica vienen del embedding layer)
# Cada token es un vector de 8 dimensiones
x = torch.tensor([
    [1.0, 0.5, 0.2, 0.8, 0.1, 0.4, 0.3, 0.9],   # token 1
    [0.2, 0.9, 0.4, 0.1, 0.7, 0.3, 0.8, 0.5],   # token 2
    [0.6, 0.1, 0.9, 0.4, 0.2, 0.7, 0.5, 0.3],   # token 3
    [0.3, 0.7, 0.2, 0.9, 0.5, 0.6, 0.1, 0.8],   # token 4
])

print(f"Forma del input: {x.shape}")  # torch.Size([4, 8])
print(f"  → 4 tokens, cada uno con embedding de 8 dimensiones")

Forma del input: torch.Size([4, 8])
  → 4 tokens, cada uno con embedding de 8 dimensiones


In [ ]:
import torch.nn as nn

d_model = 8 # dimensión del embedding

# Tres proyecciones learnable (matrices de pesos)
# bias=False para simplificar — la versión original usa bias pero no es esencial
W_q = nn.Linear(d_model, d_model, bias=False)
W_k = nn.Linear(d_model, d_model, bias=False)
W_v = nn.Linear(d_model, d_model, bias=False)

# Aplicar las proyecciones a x
# Cada token se vuelve 3 "vistas" de sí mismo: Q, K, V
Q = W_q(x)  # query: lo que cada token "pregunta"
K = W_k(x)  # key: lo que cada token "ofrece"
V = W_v(x)  # value: el contenido de cada token

print(f"Q shape: {Q.shape}")  # torch.Size([4, 8])
print(Q)
print(f"K shape: {K.shape}")
print(K)
print(f"V shape: {V.shape}")
print(V)


Q shape: torch.Size([4, 8])
tensor([[ 0.3970, -0.6113, -0.3928,  0.5497, -0.2450,  0.8979,  0.1172, -0.5173],
        [ 0.0236, -0.2994,  0.0230,  0.6650, -0.1695,  0.7332,  0.2924, -0.2103],
        [ 0.1260, -0.0461, -0.2192,  0.5792,  0.0462,  0.3461, -0.0397, -0.3056],
        [ 0.2488, -0.5099, -0.5014,  0.6193, -0.2423,  0.8382,  0.3227, -0.3829]],
       grad_fn=<MmBackward0>)
K shape: torch.Size([4, 8])
tensor([[ 0.0515, -0.3626, -0.3772, -0.1713, -0.1054,  0.0642,  0.0062,  0.3730],
        [ 0.2358, -0.0164, -0.2675,  0.3078, -0.8656,  0.2851, -0.0906,  0.2519],
        [ 0.5483,  0.0882, -0.5126, -0.3584, -0.1940,  0.0246, -0.1516,  0.3772],
        [ 0.0371, -0.3871, -0.5732,  0.2193, -0.4274, -0.0406, -0.0562,  0.1788]],
       grad_fn=<MmBackward0>)
V shape: torch.Size([4, 8])
tensor([[ 0.4283,  0.5525, -0.0289, -0.6631,  0.1729,  0.4779,  0.1718, -0.4213],
        [-0.1223,  0.5988,  0.3628, -0.1606,  0.0627,  0.6250, -0.0986, -0.4799],
        [ 0.0408,  0.2753, -0.2014

In [3]:
import torch.nn.functional as F
import math

# Attention scores: Q @ K^T
# Q: (4, 8), K.transpose: (8, 4) → scores: (4, 4)
scores = Q @ K.T

# Scaling: dividimos por sqrt(d_k) para estabilizar el softmax
# (sin esto, con d_model grande los scores explotan y el softmax da casi one-hot)
d_k = K.shape[-1]  # 8
scores = scores / math.sqrt(d_k)


print(f"Scores shape: {scores.shape}")  # torch.Size([4, 4])
print(f"Scores matrix (raw):")
print(scores)

# Softmax: convierte cada fila en probabilidades (suman 1)
attn = F.softmax(scores, dim=-1)

print(f"\nAttention weights (después de softmax):")
print(attn)
print(f"\nSuma de cada fila (debe ser 1.0): {attn.sum(dim=-1)}")

Scores shape: torch.Size([4, 4])
Scores matrix (raw):
tensor([[-0.1410, -0.0631, -0.1012, -0.1895],
        [ 0.0544,  0.0760,  0.0336,  0.0439],
        [-0.0426,  0.0354, -0.0285, -0.0605],
        [-0.0271, -0.0808, -0.0116, -0.1282]], grad_fn=<DivBackward0>)

Attention weights (después de softmax):
tensor([[0.2454, 0.2653, 0.2554, 0.2338],
        [0.2506, 0.2560, 0.2454, 0.2480],
        [0.2452, 0.2651, 0.2487, 0.2409],
        [0.2586, 0.2451, 0.2626, 0.2337]], grad_fn=<SoftmaxBackward0>)

Suma de cada fila (debe ser 1.0): tensor([1.0000, 1.0000, 1.0000, 1.0000], grad_fn=<SumBackward1>)


In [4]:
output = attn @ V

print(f"Output shape: {output.shape}")  # torch.Size([4, 8])
print(f"\nOutput[0] (representación contextualizada del token 1):")
print(output[0])

Output shape: torch.Size([4, 8])

Output[0] (representación contextualizada del token 1):
tensor([-0.1749, -0.0629, -0.0978,  0.0556,  0.3055, -0.2076, -0.1771, -0.5248],
       grad_fn=<SelectBackward0>)


## Multi-Head Attention — toy example with 2 heads

Single-head attention projects Q/K/V into one space and computes one attention pattern per token pair.

Multi-head attention runs `num_heads` independent attention subspaces in parallel. Each head works in a smaller `head_dim = d_model // num_heads` space, learns its own slice of W_q/W_k/W_v, and produces its own attention pattern. Concatenating the per-head outputs and projecting with `W_o` lets the model attend to information from different representation subspaces at different positions (Vaswani et al., 2017).

In our toy example we use `num_heads = 2` because `d_model = 8` divides cleanly into `head_dim = 4`. The original Transformer paper used 8 heads with `d_model = 512` (`head_dim = 64`); the number of heads is an architectural hyperparameter, not a universal constant. Pick it so `d_model % num_heads == 0` and so each `head_dim` is large enough for the per-head subspace to be meaningful.

In [8]:
# Split Q, K, V into num_heads independent subspaces.
# view(seq_len, num_heads, head_dim) splits the last dim into (num_heads, head_dim);
# transpose(0, 1) moves the heads dimension to the front so we get [num_heads, seq_len, head_dim].
# .contiguous() is required after transpose because the next view needs a contiguous tensor.
num_heads = 2
head_dim = d_model // num_heads
seq_len = Q.shape[0]

Q_h = Q.contiguous().view(seq_len, num_heads, head_dim).transpose(0, 1)
K_h = K.contiguous().view(seq_len, num_heads, head_dim).transpose(0, 1)
V_h = V.contiguous().view(seq_len, num_heads, head_dim).transpose(0, 1)

print(f"Q_h shape: {Q_h.shape}")
print(f"K_h shape: {K_h.shape}")
print(f"V_h shape: {V_h.shape}")
assert Q_h.shape == (num_heads, seq_len, head_dim) == (2, 4, 4)
assert K_h.shape == (2, 4, 4)
assert V_h.shape == (2, 4, 4)

Q_h shape: torch.Size([2, 4, 4])
K_h shape: torch.Size([2, 4, 4])
V_h shape: torch.Size([2, 4, 4])


In [9]:
# Per-head scaled dot-product attention.
# Each head computes its own scores with its own head_dim, so the sqrt uses head_dim (not d_model).
scores_h = Q_h @ K_h.transpose(-2, -1) / math.sqrt(head_dim)
attn_h = F.softmax(scores_h, dim=-1)

print(f"scores_h shape: {scores_h.shape}")
print(f"scores_h:\n{scores_h}")

row_sums = attn_h.sum(dim=-1)
print(f"\nattn_h row sums (every row must equal 1.0): {row_sums}")
assert torch.allclose(row_sums, torch.ones_like(row_sums))

scores_h shape: torch.Size([2, 4, 4])
scores_h:
tensor([[[ 0.1480,  0.1890,  0.0841,  0.2986],
         [-0.0064,  0.1045, -0.1318,  0.1247],
         [ 0.0033,  0.1337, -0.0151,  0.1376],
         [ 0.1404,  0.1959,  0.0633,  0.3149]],

        [[-0.0544,  0.1636, -0.0716, -0.0154],
         [-0.0058,  0.1381, -0.0364, -0.0057],
         [-0.0485, -0.0074, -0.0549, -0.0431],
         [-0.0307,  0.1615, -0.0629, -0.0085]]], grad_fn=<DivBackward0>)

attn_h row sums (every row must equal 1.0): tensor([[1.0000, 1.0000, 1.0000, 1.0000],
        [1.0000, 1.0000, 1.0000, 1.0000]], grad_fn=<SumBackward1>)


In [10]:
# Weighted sum of V_h per head: out_h[h, t, :] = sum over s of attn_h[h, t, s] * V_h[h, s, :]
out_h = attn_h @ V_h

print(f"out_h shape: {out_h.shape}")

# Indexing is zero-based: out_h[head_index][token_index] is the contextualized vector for
# that token under that head. Here we print token 0 under both heads to compare subspaces.
for h in range(num_heads):
    print(f"\nout_h[{h}][0] (head {h}, token 0):")
    print(out_h[h][0])
assert out_h.shape == (num_heads, seq_len, head_dim) == (2, 4, 4)

out_h shape: torch.Size([2, 4, 4])

out_h[0][0] (head 0, token 0):
tensor([ 0.0861,  0.5005,  0.0769, -0.4568], grad_fn=<SelectBackward0>)

out_h[1][0] (head 1, token 0):
tensor([ 0.1406,  0.4825,  0.0739, -0.3858], grad_fn=<SelectBackward0>)


In [11]:
# Concatenate per-head outputs back to [seq_len, d_model] and apply the output projection W_o.
# transpose(0, 1) puts heads back next to the feature dim; contiguous() lets view merge them.
concat = out_h.transpose(0, 1).contiguous().view(seq_len, d_model)
W_o = nn.Linear(d_model, d_model, bias=False)
final_output = W_o(concat)

print(f"concat shape: {concat.shape}")
print(f"final output shape: {final_output.shape}")
assert concat.shape == (seq_len, d_model) == (4, 8)
assert final_output.shape == (seq_len, d_model) == (4, 8)

concat shape: torch.Size([4, 8])
final output shape: torch.Size([4, 8])
